In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Scrape the data into a data set
import requests
from bs4 import BeautifulSoup


categories = [
    ("Travel", "https://books.toscrape.com/catalogue/category/books/travel_2/index.html"),
    ("Mystery", "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"),
    ("Historical Fiction", "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html")
]

data = []

for category, url in categories:

    while True:

        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        books = soup.find_all("article", class_="product_pod")

        for book in books:

            title = book.h3.a["title"]

            price = book.find("p", class_="price_color").text

            star_rating = book.find("p", class_="star-rating")["class"][1]

            availability = book.find(
                "p",
                class_="instock availability"
            ).text.strip()

            data.append([
                title,
                price,
                star_rating,
                availability,
                category
            ])

        next_page = soup.find("li", class_="next")

        if next_page:
            next_link = next_page.a["href"]

            if "index.html" in url:
                url = url.replace("index.html", next_link)
            else:
                url = url.rsplit("/", 1)[0] + "/" + next_link
        else:
            break



In [4]:
# Create the data set using the scraped data

import pandas as pd
df = pd.DataFrame(
    data,
    columns=[
        "title",
        "price",
        "star_rating",
        "availability",
        "category"
    ]
)

df.to_csv("books.csv", index=False)

print("Total books:", len(df))
df.head()


Total books: 69


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


Clean the scraped fields into proper types:

Strip the currency symbol from price and convert it to a float column price_gbp.
Convert the text star rating (One…Five) into an integer column rating (1–5).
Parse the availability text into a boolean column in_stock.
If any field fails to parse for a given row (e.g., unexpected text), handle it with the median-imputation approach for numeric fields or drop the row (state and justify your choice) — do not leave the pipeline crashing on messy rows.

In [5]:
# Copy the extracted dataset to a new variable
df_books = df.copy()
df_books.head()

# Cleaning the scraped files into proper types
# Strip the currency symbol from price and make it float   
df_books['price'] = df_books['price'].str.strip("Â£").astype(float)
df_books['price']

# Rename the column to price_gbp
df_books= df_books.rename(columns={'price': 'price_gbp'})

#Convert the text star rating (One…Five) into an integer column rating (1–5).

ratings = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df_books["rating"] = df_books["star_rating"].map(ratings)
print(df_books[["star_rating", "rating"]])

# Parse the availability text into a boolean column in_stock.
df_books = df_books.dropna(subset=["availability"])
df_books["in_stock"] = df_books["availability"].str.contains("In stock")
df_books




   star_rating  rating
0          Two       2
1         Four       4
2        Three       3
3          Two       2
4        Three       3
..         ...     ...
64        Five       5
65       Three       3
66       Three       3
67        Four       4
68        Five       5

[69 rows x 2 columns]


,title,price_gbp,star_rating,availability,category,rating,in_stock
0,It's Only the Himalayas,45.17,Two,In stock,Travel,2,True
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,Four,In stock,Travel,4,True
2,See America: A Celebration of Our National Par...,48.87,Three,In stock,Travel,3,True
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,Two,In stock,Travel,2,True
4,Under the Tuscan Sun,37.33,Three,In stock,Travel,3,True
...,...,...,...,...,...,...,...
64,While You Were Mine,41.32,Five,In stock,Historical Fiction,5,True
65,The Secret Healer,34.56,Three,In stock,Historical Fiction,3,True
66,Starlark,25.83,Three,In stock,Historical Fiction,3,True
67,Lost Among the Living,27.70,Four,In stock,Historical Fiction,4,True


Convert price_gbp to a price_inr column using the project's fixed baseline conversion rate: 1 GBP = 105.50 INR. This is an artificial, project-defined constant for this assignment, not a live or historical market rate, so it never needs a lookup or a date reference. This fixed-rate conversion is the required, keyless baseline and is what gets graded for this task — it requires no external API call and no network access; simply state this exact rate in your

In [6]:
# Convert price_gbp to a price_inr column 
# using the project's fixed baseline conversion rate: 1 GBP = 105.50 INR

df_books["price_inr"] = df_books["price_gbp"] * 105.50

df_books

,title,price_gbp,star_rating,availability,category,rating,in_stock,price_inr
0,It's Only the Himalayas,45.17,Two,In stock,Travel,2,True,4765.435
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,Four,In stock,Travel,4,True,5214.865
2,See America: A Celebration of Our National Par...,48.87,Three,In stock,Travel,3,True,5155.785
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,Two,In stock,Travel,2,True,3897.170
4,Under the Tuscan Sun,37.33,Three,In stock,Travel,3,True,3938.315
...,...,...,...,...,...,...,...,...
64,While You Were Mine,41.32,Five,In stock,Historical Fiction,5,True,4359.260
65,The Secret Healer,34.56,Three,In stock,Historical Fiction,3,True,3646.080
66,Starlark,25.83,Three,In stock,Historical Fiction,3,True,2725.065
67,Lost Among the Living,27.70,Four,In stock,Historical Fiction,4,True,2922.350


Design a normalized SQLite schema with at least two tables sharing a primary/foreign key relationship, for example:

categories(category_id INTEGER PRIMARY KEY, category_name TEXT UNIQUE)
books(book_id INTEGER PRIMARY KEY, title TEXT, price_gbp REAL, price_inr REAL, rating INTEGER, in_stock INTEGER, category_id INTEGER REFERENCES categories(category_id))
(You may rename columns/tables, but the two-table PK/FK structure is required.)

In [7]:
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()
print("All tables created succesfully")


All tables created succesfully


Using Python's sqlite3 (or pandas.DataFrame.to_sql), insert your cleaned, converted data into this schema. Then write and execute at least 5 SQL queries against the database that collectively demonstrate: SELECT/WHERE, ORDER BY, LIMIT, DISTINCT, and (IN or BETWEEN) — plus at least one JOIN between your two tables (e.g., "list the 10 highest-rated books per category"). Save each query string and its output.

In [8]:
# Insert the Data
conn = sqlite3.connect('books.db')
cursor = conn.cursor()

# 0. Clear existing rows so re-running this cell doesn't duplicate data
cursor.execute("DELETE FROM books")
cursor.execute("DELETE FROM categories")

# 1. Insert unique categories using pandas to_sql
categories_df = pd.DataFrame({'category_name': df_books['category'].unique()})
categories_df.to_sql('categories', conn, if_exists='append', index=False)

# 2. Build a category_name -> category_id map
cursor.execute("SELECT category_id, category_name FROM categories")
cat_map = {name: cid for cid, name in cursor.fetchall()}

# 3. Add category_id column to the DataFrame, keep only columns the books table needs
df_to_insert = df_books.copy()
df_to_insert['category_id'] = df_to_insert['category'].map(cat_map)
df_to_insert = df_to_insert[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']]

# 4. Append into the existing books table (don't replace/create a new one)
df_to_insert.to_sql('books', conn, if_exists='append', index=False)

conn.commit()

cursor.execute("SELECT COUNT(*) FROM categories")
print(f"categories: {cursor.fetchone()[0]} rows")
cursor.execute("SELECT COUNT(*) FROM books")
print(f"books: {cursor.fetchone()[0]} rows")

conn.close()

categories: 3 rows
books: 69 rows


In [ ]:
conn = sqlite3.connect('books.db')

# 1. SELECT / WHERE / ORDER BY / LIMIT: top 5 most expensive in-stock books
q1 = "SELECT title, price_gbp FROM books WHERE in_stock = 1 ORDER BY price_gbp DESC LIMIT 5"
print(q1)
print(pd.read_sql_query(q1, conn))

# 2. DISTINCT: which star ratings appear in the data
q2 = "SELECT DISTINCT rating FROM books ORDER BY rating"
print(q2)
print(pd.read_sql_query(q2, conn))

# 3. BETWEEN: books priced between £20 and £40
q3 = "SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20 AND 40 ORDER BY price_gbp"
print(q3)
print(pd.read_sql_query(q3, conn))

# 4. IN: books rated 4 or 5 stars
q4 = "SELECT title, rating FROM books WHERE rating IN (4, 5) ORDER BY rating DESC"
print(q4)
print(pd.read_sql_query(q4, conn))

# 5. INNER JOIN: books with their category name
q5 = """
SELECT c.category_name, b.title, b.rating
FROM books b
INNER JOIN categories c ON b.category_id = c.category_id
ORDER BY c.category_name, b.rating DESC, b.title
"""
print(q5)
print(pd.read_sql_query(q5, conn))

conn.close()

In [10]:
conn = sqlite3.connect('books.db')

# Read back two of the query results using pd.read_sql
top5_df = pd.read_sql(q1, conn)
highrated_df = pd.read_sql(q4, conn)

print("Top 5 expensive in-stock books:")
print(top5_df)
print("Books rated 4 or 5:")
print(highrated_df)

# Read the full tables into DataFrames, and the SQL join result, for comparison
categories_table = pd.read_sql("SELECT * FROM categories", conn)
books_table = pd.read_sql("SELECT * FROM books", conn)
sql_join_df = pd.read_sql(q5, conn).reset_index(drop=True)

conn.close()

# Reproduce the join using pandas merge, no SQL
merged_df = pd.merge(books_table, categories_table, on="category_id")
merged_df = merged_df[["category_name", "title", "rating"]].sort_values(
    ["category_name", "rating", "title"], ascending=[True, False, True]
).reset_index(drop=True)

print("SQL JOIN result:")
print(sql_join_df)
print("pandas merge result:")
print(merged_df)
print("Equivalent:", merged_df.equals(sql_join_df))
merged_df

Top 5 expensive in-stock books:
                                               title  price_gbp
0                      Boar Island (Anna Pigeon #19)      59.48
1  The No. 1 Ladies' Detective Agency (No. 1 Ladi...      57.70
2                   A Year in Provence (Provence #1)      56.88
3                                The Past Never Ends      56.50
4                   The Last Painting of Sara de Vos      55.55
Books rated 4 or 5:
                                                title  rating
0                  1,000 Places to See Before You Die       5
1              A Time of Torment (Charlie Parker #14)       5
2   What Happened on Beale Street (Secrets of the ...       5
3   The Bachelor Girl's Guide to Murder (Herringfo...       5
4                   The Silkworm (Cormoran Strike #2)       5
5                                   The Girl You Lost       5
6             A Flight of Arrows (The Pathfinders #2)       5
7                                        Mrs. Houdini       5
8     

,category_name,title,rating
0,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5
1,Historical Fiction,A Spy's Devotion (The Regency Spies of London #1),5
2,Historical Fiction,Between Shades of Gray,5
3,Historical Fiction,Mrs. Houdini,5
4,Historical Fiction,The Passion of Dolssa,5
...,...,...,...
64,Travel,A Summer In Europe,2
65,Travel,It's Only the Himalayas,2
66,Travel,Vagabonding: An Uncommon Guide to the Art of L...,2
67,Travel,The Great Railway Bazaar,1
